Generate the monthly NEE mean 2001-2015 for every MSA
reuse code from raster_distribution_check_time but adding back the removed MSAs and discard downscale

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import rasterstats as rstats
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
from rasterio.transform import Affine
minx, miny, maxx, maxy = -179.975, -89.975, 179.975, 89.975 # from fluxx .nc info
resolution_x = 0.05
resolution_y = 0.05
transform = rasterio.transform.from_origin(minx, maxy, resolution_x, resolution_y)
NEE_FLUXX_TRANSFORM = Affine(*transform)

In [16]:
NEE_FLUXX_TRANSFORM

Affine(0.05, 0.0, -179.975,
       0.0, -0.05, 89.975)

In [11]:

year_month = '2012-10'
fluxx_nee_file_path = f'../gis/NEE_FLUX-X/NEE_{year_month[:4]}_005_monthly.nc'
ds = xr.open_dataset(fluxx_nee_file_path)

In [13]:
ds.geospatial_lat_max 

np.float64(89.975)

In [3]:
# Function to compute mean NEE for each MSA
def calculate_msa_mean(raster_file, msa_gdf):
    mean_values = []
    for _, row in msa_gdf.iterrows():
        msa_geometry = [row['geometry']]
        with rasterio.open(raster_file) as src:
            out_image, _ = mask(src, msa_geometry, crop=True, nodata=np.nan)
            out_image = out_image[0]
        
        # Compute mean ignoring NaNs
        mean_value = np.nanmean(out_image) if np.any(~np.isnan(out_image)) else np.nan
        mean_values.append(mean_value)
    return mean_values

def calc_MSA_NEE_mean(year_month, msa_gdf_crs4326, raster_transform):
    '''
    Param:
        year_month(str): e.g.'200101'
    Return:
        dict '{nee_mean_{year_month}': list}
    '''
    print(f'Processing {year_month}...')

    # get original NEE value for specific year month
    # fluxx_nee_file_path = f'../gis/NEE/NEE.RS.FP-NONE.MLM-ALL.METEO-NONE.4320_2160.monthly.{year_month[:4]}.nc'
    fluxx_nee_file_path = f'../gis/NEE_FLUX-X/NEE_{year_month[:4]}_005_monthly.nc'

    variable = 'NEE'
    ds = xr.open_dataset(fluxx_nee_file_path)
    fluxx_nee = ds[variable][int(year_month[4:]) - 1].values  # Extract data for the specified month

    nee_orig_msa_mean = []

    # calculate mean original NEE for each MSA
    for index, row in msa_gdf_crs4326.iterrows():
        shape_info = row['geometry']
        name = row['NAMELSAD']
        # go through all geometries and compute zonal statistics
        res = rstats.zonal_stats(shape_info, fluxx_nee, affine=raster_transform, stats="mean")
        mean = res[0]['mean']
        # print(name, mean)
        nee_orig_msa_mean.append(mean)

    msa_mean_dict = {
        f'nee_mean_{year_month}': nee_orig_msa_mean
    }

    return msa_mean_dict

import constants

# ====== prepare files and parameters like crs =====
msa_gdf_crsaea = gpd.read_file(constants.MSA_REGION_FILE)
nee_crs = constants.NEE_CRS

# make sure msa and nee are in same crs
msa_gdf_crs4326 = msa_gdf_crsaea.to_crs(nee_crs)

# ====== calculate =====
# add the mean values to the msa dataframe
msa_nee_mean = msa_gdf_crsaea[['NAMELSAD']]
for year in range(2001, 2022):
    for month in range(1, 13):
        year_month = f'{year}{month:02}'
        msa_mean_dict = calc_MSA_NEE_mean(year_month, msa_gdf_crs4326, NEE_FLUXX_TRANSFORM)
        for key, value in msa_mean_dict.items():
            msa_nee_mean[key] = value
            
msa_nee_mean.to_csv('../data/intermedia/msa_nee_mean_fluxx.csv', index=False)

Processing 200101...


c:\Users\qifanw\Documents\code\.venv\Lib\site-packages\rasterstats\io.py:335: NodataWarning: Setting nodata to -999; specify nodata explicitly
  warnings.warn(
C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_27364\241599070.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_nee_mean[key] = value


Processing 200102...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_27364\241599070.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msa_nee_mean[key] = value


Processing 200103...


...

C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_27364\241599070.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_nee_mean[key] = value


Processing 202111...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_27364\241599070.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_nee_mean[key] = value


Processing 202112...


C:\Users\qifanw\AppData\Local\Temp\1\ipykernel_27364\241599070.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  msa_nee_mean[key] = value


In [ ]:
msa_nee_mean = pd.read_csv(constants.NEE_MSA_MONTHLY_MEAN)
msa_nee_mean

,NAMELSAD,nee_mean_200101,nee_mean_200102,nee_mean_200103,nee_mean_200104,nee_mean_200105,nee_mean_200106,nee_mean_200107,nee_mean_200108,nee_mean_200109,...,nee_mean_201503,nee_mean_201504,nee_mean_201505,nee_mean_201506,nee_mean_201507,nee_mean_201508,nee_mean_201509,nee_mean_201510,nee_mean_201511,nee_mean_201512
0,"Sioux Falls, SD-MN Metro Area",0.196938,0.165794,0.336959,0.803317,0.908529,0.005747,-2.841552,-2.162536,0.460029,...,0.546676,0.916681,0.923087,-0.524932,-4.111958,-2.425736,0.510565,0.898531,0.499460,0.246163
1,"Wichita Falls, TX Metro Area",0.039236,-0.095023,-0.329461,-1.127869,-0.579934,-0.121026,0.117073,0.234175,0.150717,...,-0.380534,-1.163378,-0.802617,-0.693899,-0.662105,-0.206197,0.124502,0.225531,0.109438,0.085882
2,"College Station-Bryan, TX Metro Area",0.081037,0.056323,-0.402680,-0.835971,-1.181810,-0.898453,-1.104605,-0.280526,-0.600064,...,-0.022564,-0.942450,-0.768140,-1.270416,-1.015199,-0.135981,0.243083,0.376092,0.095403,0.115941
3,"Grand Forks, ND-MN Metro Area",0.086549,0.065804,0.160914,0.474690,0.433795,-1.002750,-2.751204,-0.625679,0.499341,...,0.272592,0.632968,0.478454,-1.428197,-2.637510,-0.551793,0.689062,0.627717,0.376015,0.151262
4,"Sebring, FL Metro Area",-0.027864,0.060014,-0.160295,-0.453898,-0.615979,-1.115368,-1.048657,-1.476249,-0.905531,...,-0.350672,-0.949286,-1.055155,-1.107771,-1.057074,-1.168895,-0.784255,-0.496974,-0.256004,-0.102280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
378,"Baltimore-Columbia-Towson, MD Metro Area",0.463922,0.355070,0.350825,-0.341685,-1.521426,-2.340371,-2.573700,-1.838521,-0.894595,...,0.410412,-0.149720,-1.812146,-1.858476,-2.457379,-2.028286,-0.543352,0.235405,0.546823,0.525682
379,"Wausau, WI Metro Area",0.306864,0.247596,0.379391,0.576266,-1.144960,-1.693767,-2.487171,-1.811274,0.109337,...,0.418352,0.628587,-0.754591,-2.197036,-2.922208,-0.842889,-0.241514,0.855052,0.653901,0.421606
380,"Wichita, KS Metro Area",0.114795,0.111726,0.069614,-0.577163,-0.976278,-0.605948,-0.347088,0.134643,0.108414,...,-0.042978,-0.413870,-0.657561,-0.423870,-0.726330,-0.857068,-0.051101,0.294564,0.091737,0.081328
381,"Williamsport, PA Metro Area",0.832419,0.783956,0.897872,0.572998,-2.586020,-4.274259,-3.646483,-2.939299,-1.807260,...,0.794016,0.780977,-2.576494,-2.870689,-3.781047,-2.819292,-1.264966,0.713695,1.025101,0.952037
